In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from piepy.viz.plots import psychometric, reaction_time_cloud, reaction_time_dist

DATA_ROOT = Path(sys.executable).parents[2] / "data"
DATA_ROOT

### Loading the data

In [ ]:
# reading the whole data
all_data = pl.read_parquet(f"{DATA_ROOT}/260923_training_data.parquet")


# filter for subset of sessins that are in the Cetindag et al 2026 manuscript
sessions = [
            "230227_KC139_detect__no_cam_KC", 
            "230120_KC143_detect__no_cam_KC", 
            "230403_KC144_detect__no_cam_KC", 
            "230609_KC145_detect__no_cam_KC", 
            "230511_KC146_detect__no_cam_KC", 
            "240506_KC147_detect__no_cam_KC", 
            "240429_KC148_detect__no_cam_KC", 
            "240424_KC149_detect__no_cam_KC", 
            "240814_KC151_detect__no_cam_KC", 
            "240810_KC152_detect__no_cam_KC", 
            # below animals can be excluded
            "220915_KC300_detect__no_cam_KC", 
            "220928_KC301_detect__no_cam_KC", 
            "240718_KC302_detect__no_cam_KC", 
            "240711_KC303_detect__no_cam_KC", 
            "240807_KC304_detect__no_cam_KC", 
            ]

subset_data = all_data.filter(pl.col("sessiondir").is_in(sessions))
subset_data

### Plot the psychometric

In [ ]:
# get rid of early data
nonearly_data = subset_data.filter(pl.col("outcome")!="early")


# pr is a PlotResult custom class that carries the data, statistics and the figure
pr = psychometric(data=nonearly_data,
                  x="contrast",      # change to "contrast" if you want to pool together contralateral and ipsilateral sides
                  outcome="outcome",
                  success="hit",
                  compare="stim_type",
                  average_over="animalid",  # change to None if you want to pool all the trials from the animals
                  fit_curve=False,          # curve fitting is not good for ipsi and contra sides together, change to True when using x="contrast" above
                  palette={"0.04cpd_8.0Hz":"#E08103",
                           "0.16cpd_0.5Hz":"#018EC6"},
                  )

print(pr.data)

ax = pr.figure[1]
ax.set_xscale("symlog")
ax.set_xticks(subset_data["contrast"].drop_nulls().unique().sort().to_list())
ax.set_xticklabels(subset_data["contrast"].drop_nulls().unique().sort().to_list())

### Plot the reaction time by contrast

In [ ]:
hit_data = nonearly_data.filter(pl.col("outcome")=="hit")

pr = reaction_time_cloud(data=hit_data,
                         x="contrast",
                         value="reaction_time",
                         compare="stim_type",
                         bin_width=10,
                         average_over="animalid",
                         palette={"0.04cpd_8.0Hz":"#E08103",
                                  "0.16cpd_0.5Hz":"#018EC6"},
                         # aesthetic changes
                         dodge_width=0.01,
                         violin_widths=0.03,
                         violin_linewidth=4,
                         violin_showextrema=False,
                         violin_showmedians=True,
                         width=0.01,
                         scatter_s=30,
                         scatter_linewidth=0.3,
                         scatter_edgecolor="#FFFFFF"
                         )

print(pr.data)

ax = pr.figure[1]
ax.set_xscale("symlog")

ax.set_xticks(subset_data["contrast"].drop_nulls().unique().sort().to_list())
ax.set_xticklabels(subset_data["contrast"].drop_nulls().unique().sort().to_list())

### Plot the reaction time distribution

In [ ]:
hit_data = nonearly_data.filter(pl.col("outcome")=="hit")

CONTRAST = 1.0 # change this to look at different contrasts
contrast_data = hit_data.filter(pl.col("contrast")==CONTRAST)

pr = reaction_time_dist(data=contrast_data,
                        value="reaction_time",
                        comparing="stim_type",
                        bin_width=10,
                        average_over=None,
                        palette={"0.04cpd_8.0Hz":"#E08103",
                                 "0.16cpd_0.5Hz":"#018EC6"},
                        )

print(pr.data)

ax = pr.figure[1]
ax.set_xlim([0,2000])
ax.set_ylim([0,50])